### A Demo RAG Pipeline that Integrates RNGD Chat and Embedding models with a Vector DB
---
This notebook demonstrate a simple RAG pileline using 
- `furiosa-ai/Qwen3-Embedding-8B` for embedding generation
- `furiosa-ai/Qwen2.5-0.5B-Instruct` for text generation 
- along with PostgreSQL and pgvector as a vector DB.

The pileline can optionally use `furiosa-ai/Qwen3-Reranker-8B` as a reranker.

It focuses on demonstration the integration points between the RNGD-hosted models (with a vLLM liek nterface and the application. 

It does not focus on optimization methods for Approximate Nearest Neighbor search.

![RAG flow](rag-flow.png)

In [ ]:
! pip install openai

#### Setup the embedding server at port 8001
```
docker run -it --rm \
  --device /dev/rngd:/dev/rngd \
  --security-opt seccomp=unconfined \
  --env HF_TOKEN=$HF_TOKEN \
  -v $HOME/.cache/huggingface:/root/.cache/huggingface \
  -p 8001:8001 \
  furiosaai/furiosa-llm:latest \
  serve furiosa-ai/Qwen3-Embedding-8B \
  --port 8001 \
  --devices npu:1
  ```

#### Setup the chat server at port 8000
```
docker run -it --rm \
  --device /dev/rngd:/dev/rngd \
  --security-opt seccomp=unconfined \
  --env HF_TOKEN=$HF_TOKEN \
  -v $HOME/.cache/huggingface:/root/.cache/huggingface \
  -p 8000:8000 \
  furiosaai/furiosa-llm:latest \
  serve furiosa-ai/Qwen2.5-0.5B-Instruct \
  --devices npu:0 &
```

### Sanity check working with the chat server

In [ ]:

from openai import OpenAI

base_url = os.getenv("OPENAI_BASE_URL", "http://localhost:8000/v1")
api_key = os.getenv("OPENAI_API_KEY", "EMPTY")

client = OpenAI(api_key=api_key, base_url=base_url)

def run():
    response = client.chat.completions.create(
        model="EMPTY",
        messages=[{"role": "user", "content": "What is the capital of France?"}]
    )
    return response.choices[0].message.content

print(run())


### Sanity check working with the embeddng server

In [ ]:

import os
from openai import OpenAI
# Start server with: furiosa-llm serve path/to/embedding/model

base_url = os.getenv("OPENAI_BASE_URL", "http://localhost:8001/v1")
api_key = os.getenv("OPENAI_API_KEY", "EMPTY")

client = OpenAI(base_url=base_url, api_key=api_key)

response = client.embeddings.create(
    model="embedding-model",
    input=["Text 1", "Text 2", "Text 3"],
)
# checkout the length, you'll need it later for the vector db
for data in response.data:
    embedding = data.embedding
    print(f"Index {data.index}: {len(embedding)} dimensions")

### Setup PostgreSQL and pgvector db

In [ ]:

# https://hub.docker.com/r/ramsrib/pgvector
!docker run -d --name pgvector-16 -p 5432:5432 -e POSTGRES_PASSWORD=postgres ramsrib/pgvector:16

### Setup the psycopg for Python-based interaction with the db

In [ ]:

! pip install "psycopg[binary]" pgvector

### Creating a Table for Embeddings plus the corresponing text
Create a table that stores text documents along with their embeddings:

In [ ]:
import psycopg

conn = psycopg.connect(
    "host=localhost port=5432 dbname=postgres user=postgres password=postgres",
    autocommit=True,
)

with conn.cursor() as cur:
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    cur.execute("""
        CREATE TABLE IF NOT EXISTS documents (
            id SERIAL PRIMARY KEY,
            content TEXT,
            embedding VECTOR(4096)
        );
    """)

print("Table created.")

### Inserting Embeddings with psycopq library
You can generate embeddings in Python using OpenAI or any other embedding model and insert them into PostgreSQL.

In [ ]:
import psycopg
from pgvector.psycopg import register_vector
from openai import OpenAI

embed_client = OpenAI(
    base_url="http://localhost:8001/v1",
    api_key="EMPTY",
)

conn = psycopg.connect(
    "host=localhost port=5432 dbname=postgres user=postgres password=postgres",
    autocommit=True,
)
register_vector(conn)

texts = [
    "PostgreSQL is an open-source relational database.",
    "RAG combines retrieval and generation for better answers.",
    "pgvector adds vector search capabilities to Postgres.",
]

response = embed_client.embeddings.create(
    model="embedding-model",
    input=texts,
)

with conn.cursor() as cur:
    for text, data in zip(texts, response.data):
        cur.execute(
            "INSERT INTO documents (content, embedding) VALUES (%s, %s)",
            (text, data.embedding),
        )

print(f"Inserted {len(texts)} documents.")

### Performing Semantic Retrieval
To find similar documents, you embed the user query and perform a similarity search using PostgreSQL’s vector operators.

In [ ]:
query = "How do I search vectors in Postgres?"

query_embedding = embed_client.embeddings.create(
    input=query,
    model="embedding-model",
).data[0].embedding

with conn.cursor() as cur:
    cur.execute("""
        SELECT content, embedding <-> %s::vector AS distance
        FROM documents
        ORDER BY distance ASC
        LIMIT 3;
    """, (query_embedding,))
    results = cur.fetchall()

for content, distance in results:
    print(f"{distance:.4f}  {content}")

### The generation stage the RAG Pipeline
Once you retrieve the most similar text snippets, you can feed them into an LLM as context to produce grounded responses.

In [ ]:
chat_client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",
)

context = "\n".join(r[0] for r in results)
prompt = f"""You are an assistant that answers questions using the context below.

Context:
{context}

Question: {query}
"""

response = chat_client.chat.completions.create(
    model="EMPTY",
    messages=[{"role": "user", "content": prompt}],
)

print(response.choices[0].message.content)

### Credits

https://medium.com/@nitinprodduturi/using-postgresql-as-a-vector-database-for-rag-retrieval-augmented-generation-c62cfebd9560